# Project SPECTRA — FDIA Pipeline v2
**Signal Processing + ML | Airbus FYI 2025 | IEEE TAES**

### Before running:
1. Upload `dftrain.h5`, `dfvalid.h5`, `dfvalid_groundtruth.csv` to Google Drive in a folder called `SPECTRA_data`
2. Update `DRIVE_DATA_PATH` in Cell 1 if your folder name differs
3. `Runtime → Run all`

In [ ]:
# CELL 1 — Mount Drive + verify files
from google.colab import drive
drive.mount('/content/drive')
import os

DRIVE_DATA_PATH = '/content/drive/MyDrive/SPECTRA_data'

for f in ['dftrain.h5', 'dfvalid.h5', 'dfvalid_groundtruth.csv']:
    path = os.path.join(DRIVE_DATA_PATH, f)
    size = f'{os.path.getsize(path)/1e6:.1f} MB' if os.path.exists(path) else 'MISSING'
    print(f'  {f}: {size}')

In [ ]:
# CELL 2 — Clone repo
import os
if not os.path.exists('/content/Project_SPECTRA'):
    !git clone https://github.com/Adhith-Krishna/Project_SPECTRA.git /content/Project_SPECTRA
else:
    !git -C /content/Project_SPECTRA pull
!ls /content/Project_SPECTRA/
!pip install -q h5py scikit-learn scipy numpy matplotlib

In [ ]:
# CELL 3 — Patch DATA_DIR and import pipeline as module
import sys
sys.path.insert(0, '/content/Project_SPECTRA')

with open('/content/Project_SPECTRA/fdia_pipeline.py', 'r') as f:
    src = f.read()

src = src.replace(
    'DATA_DIR  = os.path.expanduser(\n    "~/Airbus_Fly_Your_Ideas_2026/Project_Spectra/data/airbus_heli"\n)',
    f'DATA_DIR  = "{DRIVE_DATA_PATH}"'
).replace('if __name__ == "__main__":', 'if False:')

with open('/content/fdia_v2.py', 'w') as f:
    f.write(src)

import importlib
sys.path.insert(0, '/content')
import fdia_v2 as pipe
importlib.reload(pipe)
print('Imported. DATA_DIR =', pipe.DATA_DIR)

In [ ]:
# CELL 4 — Run full pipeline
detector, hardened, results, tau, tau_h = pipe.main()

In [ ]:
# CELL 5 — Save all outputs to Drive
import shutil
OUT = os.path.join(DRIVE_DATA_PATH, 'results_v2')
os.makedirs(OUT, exist_ok=True)
for fname in ['raw_signals.png','feature_distributions.png',
              'roc_curves.png','score_distributions.png']:
    if os.path.exists(fname):
        shutil.copy(fname, os.path.join(OUT, fname))
        print(f'Saved: {fname}')
print(f'Outputs in Drive/SPECTRA_data/results_v2/')

In [ ]:
# CELL 6 — Spot check: score individual sequences
import numpy as np, h5py, csv

with h5py.File(os.path.join(DRIVE_DATA_PATH,'dfvalid.h5'),'r') as f:
    test_signals = np.array(f['dfvalid/block0_values'], dtype=np.float32)

labels = np.zeros(594, dtype=np.int32)
with open(os.path.join(DRIVE_DATA_PATH,'dfvalid_groundtruth.csv')) as f:
    for row in csv.DictReader(f):
        labels[int(float(row['seqID']))] = int(float(row['anomaly']))

print(f'tau={tau:.4f}  tau_hardened={tau_h:.4f}\n')
for tag, idx_arr in [('HEALTHY', np.where(labels==0)[0][:5]),
                      ('ANOMALOUS', np.where(labels==1)[0][:5])]:
    print(f'{tag}:')
    for idx in idx_arr:
        feat = pipe.extract_sequence_features(test_signals[idx])
        s    = detector.score(feat.reshape(1,-1))[0]
        sh   = hardened.score(feat.reshape(1,-1))[0]
        flag = 'DETECTED' if s > tau else 'missed'
        print(f'  seqID={idx}  score={s:.4f} [{flag}]  hardened={sh:.4f}')
    print()